# MotifBank Demo
**Fragment energy caching for quantum chemistry — 52×–851× speedup on zeolites and MOFs**

This notebook demonstrates MotifBank in **mock QC mode** (no PySCF needed).  
All geometry, phase classification, and speedup calculations are real.  
Energy values use a mock function (random but deterministic per geometry).

GitHub: https://github.com/yoiyoicarnival/motifbank

---
**Run time: ~2 minutes (Colab CPU)**

In [ ]:
# Install dependencies (PySCF not required for this demo)
!pip install -q numpy ase
!git clone -q https://github.com/yoiyoicarnival/motifbank
import os, sys
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, 'motifbank')
print('Ready.')

## Step 1: Load a real zeolite CIF and classify its phase

MotifBank first runs a **phase classifier** to predict speedup before any QC calculation.

In [ ]:
from motifbank_cli import from_cif, classify
import numpy as np

materials = {
    'ice Ih':          ('motifbank/examples/ice_Ih_3x3.cif', (1,1,1), 'h2o'),
    'α-cristobalite':  ('motifbank/examples/cristobalite_alpha.cif', (2,2,1), 'si_oh4'),
    'LTA zeolite':     ('motifbank/examples/LTA_iza.cif', (1,1,1), 'si_oh4'),
    'MFI silicalite':  ('motifbank/examples/MFI_iza.cif', (1,1,1), 'si_oh4'),
}

print(f'{'Material':22s}  N     N_bank  compression  reuse   Phase  strategy')
print('-'*75)

for name, (cif, sc, mtype) in materials.items():
    mols, _, _ = from_cif(cif, supercell=sc, mol_type=mtype, verbose=False)
    r = classify(mols)
    N = len(mols)
    nb = r.get('n_bank', 1)
    comp = N / nb if nb > 0 else 0
    reuse = r.get('roi_second', r.get('roi_pct', 0)) / 100
    phase = r.get('phase', '?')
    strat = r.get('strategy', '')
    print(f'{name:22s}  {N:<5d} {nb:<7d} {comp:6.0f}×      {reuse:.0%}   Phase {phase}  {strat}')

## Step 2: Run MotifBank vs naive MBE and verify ΔE = 0

Both approaches compute the same total energy — MotifBank just caches and reuses geometrically identical fragments.

In [ ]:
from motifbank_cli import MotifBank, geom_key, qc_compute_mock, run_mbe
import time

# Use LTA zeolite as demonstration
mols, atypes, _ = from_cif('motifbank/examples/LTA_iza.cif', supercell=(1,1,1), mol_type='si_oh4', verbose=False)

print(f'System: LTA zeolite, N = {len(mols)} SiO4 fragments')
print()

# Naive: compute every fragment independently
t0 = time.time()
e_naive = [qc_compute_mock(mol) for mol in mols]
E_naive = sum(e_naive)
t_naive = time.time() - t0
print(f'Naive   : {len(mols)} QC calls  E = {E_naive:.6f}  ({t_naive*1000:.0f} ms mock)')

# MotifBank: cache and reuse
bank = MotifBank(path=None)
t0 = time.time()
e_bank_list = []
new_calls = 0
for mol in mols:
    k = geom_key(mol)
    cached = bank.query_exact(k)
    if cached is not None:
        e_bank_list.append(cached)
    else:
        e = qc_compute_mock(mol)
        bank.store(k, mol, e)
        e_bank_list.append(e)
        new_calls += 1
E_bank = sum(e_bank_list)
t_bank = time.time() - t0
print(f'MotifBank: {new_calls} QC calls   E = {E_bank:.6f}  ({t_bank*1000:.0f} ms mock)')

dE = abs(E_naive - E_bank)
speedup = len(mols) / new_calls
print()
print(f'ΔE      = {dE:.2e} Ha  →  {"EXACT MATCH ✓" if dE < 1e-8 else "MISMATCH ✗"}')
print(f'Speedup = {speedup:.0f}× (fragment call reduction)')

## Step 3: Speedup scaling — how it grows with system size

For Phase 0/1 materials, N_bank saturates → speedup grows indefinitely.

In [ ]:
import matplotlib.pyplot as plt

# MFI silicalite-1: measured N_bank = 282 (saturated)
N_bank_mfi = 282
T_QC = 26.7  # seconds per PySCF PBE/def2-SVP call (measured)

# Approximate total QC calls for MBE: monomers + pairs + trimers within R_cut
# From benchmark: 14,690 calls at N=768
N_vals = [96, 192, 384, 768, 1536, 3000, 6000, 10000]
# Scale proportionally from measured 768→14690
naive_calls = [int(14690 * n / 768) for n in N_vals]
bank_calls  = [N_bank_mfi] * len(N_vals)
speedups    = [nc / N_bank_mfi for nc in naive_calls]
t_naive_h   = [nc * T_QC / 3600 for nc in naive_calls]
t_bank_h    = [N_bank_mfi * T_QC / 3600] * len(N_vals)

print('MFI silicalite-1  (PBE/def2-SVP, T_QC=26.7s, N_bank_sat=282)')
print(f'{"N":>7}  {"naive calls":>12}  {"bank calls":>10}  {"speedup":>8}  {"T_naive":>10}  {"T_bank":>8}')
print('-'*65)
for i, N in enumerate(N_vals):
    print(f'{N:>7}  {naive_calls[i]:>12,}  {bank_calls[i]:>10}  {speedups[i]:>7.0f}×  {t_naive_h[i]:>8.1f}h  {t_bank_h[i]:>6.2f}h')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(N_vals, speedups, 'o-', color='steelblue', linewidth=2, markersize=6)
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('System size N (SiO4 units)', fontsize=12)
axes[0].set_ylabel('Speedup (×)', fontsize=12)
axes[0].set_title('MFI silicalite-1: MotifBank speedup vs system size', fontsize=11)
axes[0].grid(True, alpha=0.3)
for i, (n, s) in enumerate(zip(N_vals, speedups)):
    if n in [768, 10000]:
        axes[0].annotate(f'{s:.0f}×', (n, s), textcoords='offset points', xytext=(5, 5), fontsize=9)

axes[1].semilogy(N_vals, t_naive_h, 'o-', color='red', label='Naive MBE', linewidth=2)
axes[1].semilogy(N_vals, t_bank_h, 's--', color='green', label='MotifBank', linewidth=2)
axes[1].set_xlabel('System size N (SiO4 units)', fontsize=12)
axes[1].set_ylabel('Wall time (hours, log scale)', fontsize=12)
axes[1].set_title('Wall time comparison (PBE/def2-SVP)', fontsize=11)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('motifbank_scaling_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: motifbank_scaling_demo.png')

## Step 4: Phase diagram — which materials benefit most?

In [ ]:
# S_local = log(N_bank_sat) — information entropy of local geometry
import numpy as np
import matplotlib.pyplot as plt

data = [
    ('ice Ih',          16,   0.086, 'Phase 0', '#2196F3'),
    ('α-cristobalite',  18,   0.091, 'Phase 0', '#2196F3'),
    ('LTA zeolite',     66,   0.177, 'Phase 0', '#4CAF50'),
    ('MFI silicalite',  282,  0.320, 'Phase 0', '#FF9800'),
]

names   = [d[0] for d in data]
n_banks = [d[1] for d in data]
s_local = [np.log(n) for n in n_banks]
colors  = [d[4] for d in data]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, s_local, color=colors, edgecolor='white', linewidth=0.5)
for bar, nb in zip(bars, n_banks):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'N_bank={nb}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('S_local = log(N_bank_sat)  [nats]', fontsize=11)
ax.set_title('Local geometric entropy of periodic materials\n(lower = more reusable = higher MotifBank speedup)', fontsize=10)
ax.set_ylim(0, 7)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('s_local_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nS_local interpretation:')
print('  Low S_local → few unique geometries → MotifBank gives maximum speedup')
print('  ice Ih  (S=2.77): speedup → ∞ as N → ∞')
print('  MFI     (S=5.64): speedup → 851× at N=10,000')

## Summary

| Material | N_bank_sat | S_local | Speedup (N=768) |
|----------|-----------|---------|------------------|
| ice Ih | 16 | 2.77 nats | ~108× |
| α-cristobalite | 18 | 2.89 nats | ~148× |
| LTA zeolite | 66 | 4.19 nats | ~135× |
| MFI silicalite-1 | 282 | 5.64 nats | **52×** |

**Accuracy: ΔE = 0.00 Ha (exact, verified PBE/def2-SVP)**

### Install and use

```bash
pip install numpy ase "pyscf>=2.0" fastapi uvicorn
git clone https://github.com/yoiyoicarnival/motifbank
cd motifbank
OMP_NUM_THREADS=1 python3 motifbank_cli.py demo
```

GitHub: https://github.com/yoiyoicarnival/motifbank